In [ ]:
import pandas as pd
import numpy as np

---

## Level 1 — Daily temperature trend

**New concept: `ewm()`**

`ewm()` — exponentially weighted moving window — gives more weight to recent observations and less to older ones:

```python
s.ewm(span=7).mean()   # weighted mean — recent rows count more
s.ewm(span=7).std()    # weighted std — adapts to recent volatility
```

`span` controls how fast older values decay. Small span (e.g. 3) = reactive, tracks changes quickly. Large span (e.g. 20) = smooth, slow-moving trend.

Compare to `expanding().mean()`, which gives *equal* weight to every row so far — it's anchored to history and slow to react to recent shifts.

---

A weather station logs daily temperature (°C) for 30 days. Temperatures warm over the month, but day-to-day readings are noisy.

1. Add `ewm_mean = temp_c.ewm(span=7).mean()` and `exp_mean = temp_c.expanding().mean()`. Print the first 10 rows of all three columns side by side.
2. On which day is the absolute gap between `ewm_mean` and `exp_mean` largest? Use `np.abs` on the difference, then `np.argmax`.
3. `np.corrcoef` on `ewm_mean` and `exp_mean` — how similar are the two smoothers overall?

In [ ]:
weather = pd.DataFrame({
    'date':   pd.date_range('2023-06-01', periods=30),
    'temp_c': [21.3, 23.1, 19.8, 22.4, 24.7, 20.1, 18.9, 25.3, 23.8, 26.1,
               22.7, 27.4, 24.2, 28.9, 25.6, 21.3, 29.8, 26.4, 24.1, 30.2,
               27.8, 25.3, 31.4, 28.9, 26.7, 32.1, 29.4, 27.8, 33.6, 30.2],
})

# Your code here

---

## Level 2 — Call centre handle time

Three agents are tracked weekly for 12 weeks. Lower `handle_min` = faster = better. Their trajectories differ: one is steadily improving, one is all over the place, one is slowing down.

Use `ewm(span=4)` per agent to capture their recent form — not their all-time average.

1. For each agent, extract their rows and compute `ewm_handle = handle_min.ewm(span=4).mean()`. Print the **last 3 rows** per agent — what does the recent trend look like for each?
2. Collect each agent's **final** `ewm_handle` value into a plain Python list. Use `np.argsort` to rank them from best (lowest) to worst. Print the agent names in that order.
3. `np.percentile` on **all** 36 `handle_min` values — find Q1 and Q3. How many readings fall below Q1?

In [ ]:
calls = pd.DataFrame({
    'week':       list(range(1, 13)) * 3,
    'agent':      ['Ana'] * 12 + ['Ben'] * 12 + ['Cal'] * 12,
    'handle_min': [
        12.4, 11.8, 12.1, 11.5, 10.9, 11.2, 10.6, 10.1,  9.8,  9.5,  9.2,  8.9,  # Ana
        11.2, 13.5,  9.8, 14.2, 10.4, 12.9, 10.1, 13.8,  9.7, 11.4, 10.2, 12.6,  # Ben
         9.1,  9.4,  9.8, 10.2,  9.9, 10.5, 10.8, 11.2, 10.9, 11.5, 11.8, 12.3,  # Cal
    ],
})

# Your code here

---

## Level 3 — Two ad channels

`search` and `display` log weekly conversions on alternating weeks over six months. Fewer guardrails — figure out the right tools.

1. Build the full weekly timeline: `merge_ordered` + ffill + dropna.
2. Add `ewm_search = search_conv.ewm(span=4).mean()` and `ewm_display = display_conv.ewm(span=4).mean()`. At the **last week**, which channel has the higher ewm trend?
3. Add `total = search_conv + display_conv`. Use `expanding().sum()` — what's the cumulative total by the final week?
4. `np.corrcoef` on `search_conv` and `display_conv`. Do the two channels move together?
5. Add `gap = ewm_search - ewm_display`. Use `np.argsort` on `np.abs(gap)` to find the 3 weeks where the channels diverged most. Which channel was ahead in those weeks?

In [ ]:
search = pd.DataFrame({
    'week':        pd.to_datetime(['2023-01-02','2023-01-16','2023-01-30','2023-02-13',
                                   '2023-02-27','2023-03-13','2023-03-27','2023-04-10',
                                   '2023-04-24','2023-05-08','2023-05-22','2023-06-05']),
    'search_conv': [1240, 1380, 1290, 1520, 1460, 1680, 1590, 1820, 1750, 1940, 1870, 2100],
})

display = pd.DataFrame({
    'week':         pd.to_datetime(['2023-01-09','2023-01-23','2023-02-06','2023-02-20',
                                    '2023-03-06','2023-03-20','2023-04-03','2023-04-17',
                                    '2023-05-01','2023-05-15','2023-05-29','2023-06-12']),
    'display_conv': [2100, 1980, 2240, 2050, 2380, 2190, 2510, 2320, 2620, 2440, 2730, 2580],
})

# Your code here